## Deep Reinforcement Learning on Grid World Assignment

### Instructions :

1. Initialisez le Grid World avec une largeur, une hauteur et des positions bloquées données.
    - Créez une grille bidimensionnelle représentant le monde avec la largeur et la hauteur spécifiées.
    - Marquez toutes les positions bloquées sur la grille.
    - Spécifiez une position de départ et une position d'arrivée sur la grille.
    - Lancez une simulation avec un agent aléatoire.

2. Créez un Replay Buffer (mémoire tampon de rejouement).
    - Implémentez une classe Replay Buffer capable de stocker et d'échantillonner des expériences pour entraîner l'agent DQN.
    - Chaque expérience doit contenir l'état courant, l'action effectuée, la récompense reçue, l'état suivant et un indicateur (done) signalant si l'épisode est terminé.

3. Créez un réseau de neurones convolutif pour l'apprentissage d'un agent DQN.
    - Définissez un CNN avec Pytorch contenant 3 couches de convolution et 2 couches linéaires.

4. Implémentez un agent DQN pour naviguer dans le Grid World.
    - Implémentez une classe d'agent DQN qui utilise un réseau de neurones pour approximer les Q-valeurs pour chaque paire état-action.
    - Utilisez une politique epsilon-greedy pour sélectionner les actions pendant l'entraînement.
    - Utilisez le replay buffer pour échantillonner des expériences et entraîner l'agent DQN à mettre à jour ses Q-valeurs.
    - Entraînez l'agent à atteindre la position d'arrivée depuis la position de départ tout en évitant les positions bloquées.

5. Expérimentation
    - Entraînez l'agent DQN à naviguer de la position de départ à la position d'arrivée tout en évitant les positions bloquées.
    - Expérimentez avec différentes valeurs d'hyperparamètres, comme le taux d'apprentissage, le facteur d'actualisation, la taille du batch, la récompense, et observez leurs effets sur l'apprentissage de l'agent DQN.
    - Analysez la performance de l'agent en mesurant le nombre d'étapes nécessaires pour atteindre l'objectif à chaque épisode.

6. Simulation
    - Visualisez le chemin de l'agent de la position de départ à la position d'arrivée.
    - Simulez l'agent avec la politique sauvegardée à un épisode spécifique.

7. (Optionnel)
    - Proposez une configuration difficile et observez la performance de l'agent DQN.

8. Questions :
    - Expliquez le rôle du réseau de neurones et la motivation de son utilisation.
    - Expliquez le rôle de la mémoire de rejouement et du minibatch.
    - Expliquez le choix des différents composants de la mémoire de rejouement et leur importance dans l'apprentissage.
    - Expliquez la relation entre la fonction de perte du DQN et la fonction de mise à jour des Q-valeurs dans le Q-Learning.


### GridWorld Environment

In [1]:
from collections import deque

In [2]:
import random
import numpy as np
class GridWorld:
    def __init__(self, width, height, blocked_positions=None):
        # Initialize the GridWorld with the given width and height
        # Set the player's starting position to (0, 0) and the goal position to (width - 1, height - 1)
        # Set the blocked positions to the given list of positions or an empty list if none is given
        # Set the observation space to be a tuple of (2, height, width) and the action space to be 4
        # Set the list of actions to be ["up", "down", "left", "right"]
        self.width = width
        self.height = height
        self.player_position = (0, 0)
        self.goal_position = (width - 1, height - 1)
        self.blocked_positions = blocked_positions if blocked_positions else []
        self.observation_space = (2, self.height, self.width)
        self.action_space = 4
        self.actions = ["up", "down", "left", "right",]
          
    def make_action(self, action:int):
        # Move the player in the given direction based on the given action
        # Call the move_player method with the corresponding direction
        self.move_player(self.actions[action])

    def move_player(self, direction):
        # Move the player in the given direction
        # Check if the new position is blocked, and if not, update the player's position
        # Return True if the player has reached the goal position, False otherwise
        x, y = self.player_position
        if direction == "up":
            y = max(0, y - 1)
        elif direction == "down":
            y = min(self.height - 1, y + 1)
        elif direction == "left":
            x = max(0, x - 1)
        elif direction == "right":
            x = min(self.width - 1, x + 1)
        
        # Check if the new position is blocked
        if (x, y) not in self.blocked_positions:
            self.player_position = (x, y)
        return self.player_position == self.goal_position

    def print_grid(self):
        # Print the current state of the grid, with the player represented by "P", the goal by "G",
        # blocked positions by "B", and empty spaces by "."
        for y in range(self.height):
            row = ""
            for x in range(self.width):
                if (x, y) == self.player_position:
                    row += "P"
                elif (x, y) == self.goal_position:
                    row += "G"
                elif (x, y) in self.blocked_positions:
                    row += "B"
                else:
                    row += "."
            print(row)
            
    def is_game_over(self):
        # Return True if the player has reached the goal position, False otherwise
        return self.player_position == self.goal_position
    
    def get_state(self):
        # Return the current state of the game as a tuple
        # The state is represented as a numpy array of shape (2, height, width)
        # The first channel represents the player's position, with a 1 at the current position and 0 elsewhere
        # The second channel represents the blocked positions, with a 1 at each blocked position and 0 elsewhere
        state = np.zeros((2, self.height, self.width))
        state[0][self.player_position] = 1
        for  block_x, block_y  in self.blocked_positions:
            state[1][block_x, block_y] = 1
        return state

    def get_reward(self):
        # Return the reward for the current state of the game
        # The reward is 100 if the player has reached the goal position, -1 otherwise
        reward = 0
        if self.is_game_over():
            reward = 1
        else:
            reward = -0.01
        return reward
    
    def reset(self):
        # Reset the player's position to (0, 0)
        self.player_position = (0, 0)

Here is an example using pygame interface for simulation of a random agent 

In [ ]:
import pygame

# Define grid cell size and margin
CELL_SIZE = 50
MARGIN = 5

# Define colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)

def draw_grid(screen, grid_world):
    for y in range(grid_world.height):
        for x in range(grid_world.width):
            color = WHITE
            if (x, y) == grid_world.player_position:
                color = RED
            elif (x, y) == grid_world.goal_position:
                color = GREEN
            elif (x, y) in grid_world.blocked_positions:
                color = BLUE

            pygame.draw.rect(screen, color, [(MARGIN + CELL_SIZE) * x + MARGIN, (MARGIN + CELL_SIZE) * y + MARGIN, CELL_SIZE, CELL_SIZE])

def main():
    grid_world = GridWorld(5, 5, blocked_positions=[(1, 1), (2, 2), (3, 3)])
    directions = ["up", "down", "left", "right"]

    # Set up the display
    screen = pygame.display.set_mode([(CELL_SIZE + MARGIN) * grid_world.width + MARGIN, (CELL_SIZE + MARGIN) * grid_world.height + MARGIN])
    pygame.display.set_caption("GridWorld")

    # Main loop
    clock = pygame.time.Clock()
    while not grid_world.is_game_over():
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        # Random agent simulation
        if not grid_world.is_game_over():
            direction = random.choice(directions)
            grid_world.move_player(direction)

        # Draw the grid
        screen.fill(BLACK)
        draw_grid(screen, grid_world)

        # Update the display
        pygame.display.flip()

        # Limit the frame rate
        clock.tick(30)

    # Quit pygame
    pygame.quit()

main()

### Deep Qlearning Agent

The Deep Q-learning agent is an extension of the Q-learning algorithm that uses a deep neural network to estimate the Q-values instead of a Q-table. This makes it possible to handle high-dimensional state spaces and large action spaces more efficiently.

The Deep Q-learning algorithm is composed of several key elements:
- The agent uses a deep neural network to estimate the Q-values for each state-action pair.
- The agent updates the network parameters after each action based on the reward it receives and the maximum Q-value of the next state.
- The agent chooses the action with the highest Q-value predicted by the neural network in each state.


The Q-value of a state-action pair is updated using the following formula:
$$Q(s_t,a_t) \leftarrow  Q(s_t,a_t) + \alpha (r_t + \gamma \max_{a'} Q(s_{t+1},a') - Q(s_t,a_t))$$

where $s$ is the current state, $a$ is the action taken in that state, $r$ is the reward received for taking that action, $s'$ is the next state, $\alpha$ is the learning rate, $\gamma$ is the discount factor. Note that $Q_{target}(s,a)=r + \gamma \max_{a'} Q(s',a')$ is the estimated target Q-value.

The agent updates the neural network parameters using the mean-squared error loss between the predicted Q-value and the target Q-value. The target Q-value is calculated as:
$$Min_{\theta} \|Q_{target}(s_t,a_t) -  Q(s_t,a_t)\| = Min_{\theta} \|r + \gamma \max_{a'} Q(s{_t+1},a') - Q(s_t,a_t)\|$$

​

In [4]:
class ReplayMemory:
    def __init__(self, capacity):
        # Initializes the ReplayMemory class with a given capacity.
        self.capacity = capacity
        self.memory = deque(maxlen=capacity)
        self.current_episode = []
        self.neg_memory = deque(maxlen=capacity)
        self.pos_trajectory_memory = deque(maxlen=capacity)
        self.pos_memory = deque(maxlen=capacity)

    def new_episode(self):
        # Clears the current episode.
        self.current_episode = []

    def push(self, state, action, reward, next_state, done):
        # Adds a new experience to the memory.
        self.memory.append((state, action, reward, next_state, done))
        self.current_episode.append((state, action, reward, next_state, done))
        if reward < -0.01:
            self.neg_memory.append((state, action, reward, next_state, done))
        if reward > 0.01:
            self.pos_memory.append((state, action, reward, next_state, done))

    def is_done(self):
        # Returns whether the current episode is done.
        return self.current_episode[-1][4]


    def get_batch(self, batch_size):
        # The function `get_batch` returns a batch of experiences from the memory. 
        # The batch is composed of three different types of experiences: `batch`, 
        # `pos_trajectory_batch`, and `pos_batch`. 

        # `batch` is a random sample of experiences from the memory. 

        # `pos_trajectory_batch` is a random sample of experiences from the `pos_trajectory_memory` deque.
        #  
        # `pos_batch` is a random sample of experiences from the `pos_memory` deque. 

        # The `pos_trajectory_memory` deque is used to store experiences that have a 
        # positive reward and are part of a trajectory that led to a positive reward. 

        # The `pos_memory` deque is used to store experiences that have a positive reward 
        # but are not part of a trajectory that led to a positive reward. 
        
        # The purpose of having separate batches for positive experiences is to ensure that 
        # the agent learns from positive experiences more frequently.

        pos_trajectory_batch = random.sample(self.pos_trajectory_memory, min(len(self.pos_trajectory_memory), batch_size//3))
        pos_batch = random.sample(self.pos_memory, min(len(self.pos_memory), batch_size//3))
        batch = random.sample(self.memory, min(len(self.memory), batch_size - len(pos_trajectory_batch) - len(pos_batch)))
        batch = batch + pos_trajectory_batch + pos_batch
        state_batch = []
        action_batch = []
        reward_batch = []
        next_state_batch = []
        done_batch = []
        for state, action, reward, next_state, done in batch:
            state_batch.append(state)
            action_batch.append(action)
            reward_batch.append(reward)
            next_state_batch.append(next_state)
            done_batch.append(done)
        return state_batch, action_batch, reward_batch, next_state_batch, done_batch

    def __len__(self):
        # Returns the length of the memory.
        return len(self.memory)
    
    def add_episode_pos(self):
        # Add positive trajectory samples into pos_trajectory_memory.
        self.pos_trajectory_memory.extend(self.current_episode)
        self.new_episode()




In [5]:
import matplotlib.pyplot as plt
import pickle
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
class DQN(nn.Module):
    def __init__(self, input_shape, num_actions):
        super(DQN, self).__init__()
        # Define the convolutional layers
        self.conv1 = nn.Conv2d(input_shape[0], 16, kernel_size=5, stride=1)
        self.dropout1 = nn.Dropout(p=0.5)
        self.conv2 = nn.Conv2d(16,32, kernel_size=4, stride=1)
        self.dropout2 = nn.Dropout(p=0.5)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1)
        self.dropout3 = nn.Dropout(p=0.5)
        # Define the fully connected layers
        self.fc1 = nn.Linear(self.feature_size(input_shape), 512)
        self.dropout4 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(512, num_actions)

    def forward(self, x):
        # Pass the input through the convolutional layers
        x = F.relu(self.conv1(x))
        x = self.dropout1(x)
        x = F.relu(self.conv2(x))
        x = self.dropout2(x)
        x = F.relu(self.conv3(x))
        x = self.dropout3(x)
        # Flatten the output and pass it through the fully connected layers
        x = F.relu(self.fc1(x.view(x.size(0), -1)))
        x = self.dropout4(x)
        return self.fc2(x)

    def feature_size(self, input_shape):
        # Compute the size of the output of the convolutional layers
        print(input_shape)
        x = self.conv1(torch.zeros(1, *input_shape))
        x = self.conv2(x)
        x = self.conv3(x)
        return x.view(1, -1).size(1)




In [6]:

class DQNAgent:
    def __init__(self, alpha=0.1, gamma=0.99, epsilon=0.1, dqn_model=None):
        """
        Initializes the DQNAgent class with the given parameters.
        """
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = {}
        self.dqn_model = dqn_model

    def get_q_value(self, state, action=None):
        """
        Returns the Q-value of the given state-action pair.
        If action is None, returns the Q-values of all actions for the given state.
        """
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = self.dqn_model(state_tensor)
        if action is None:
            return q_values
        else:
            return q_values[action]
        
    def update_epsilon(self, epsilon_decay = 0.99):
        """
        Updates the value of epsilon by multiplying it with the given decay factor.
        """
        self.epsilon *= epsilon_decay
        
    def get_action(self, state, action_size=4):
        """
        Returns the action to be taken for the given state using an epsilon-greedy policy.
        """
        if random.random() < self.epsilon:
            return random.randint(0, action_size-1)
        else:
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                q_values = self.dqn_model(state_tensor)
            return q_values.argmax().item()

    def learn_from_memory(self, memory_batch):
        """
        Updates the Q-values of the DQN model using the given batch of experiences.
        """
        state_batch, action_batch, reward_batch, next_state_batch, done_batch = memory_batch
        state_tensor = torch.tensor(state_batch, dtype=torch.float32)
        action_tensor = torch.tensor(action_batch, dtype=torch.int64).unsqueeze(1)
        reward_tensor = torch.tensor(reward_batch, dtype=torch.float32).unsqueeze(1)
        next_state_tensor = torch.tensor(next_state_batch, dtype=torch.float32)
        done_tensor = torch.tensor(done_batch, dtype=torch.float32).unsqueeze(1)

        # `torch.no_grad()` is a PyTorch context manager that disables gradient calculation. 
        # This is useful during inference or testing, where we do not need to compute gradients 
        # and can save memory and computation time. 
        # In the given code, `torch.no_grad()` is used to compute the Q-values of the DQN model 
        # for a given state without computing gradients. 
        # This is done using the `self.dqn_model` attribute of the `DQNAgent` class, 
        # which is a PyTorch neural network model. 
        # The Q-values are computed by passing the state tensor through the model using `self.dqn_model(state_tensor)`. 
        # The resulting tensor contains the Q-values for all actions in the given state.
        with torch.no_grad():
            # In PyTorch, `model.eval()` sets the model to evaluation mode. 
            # This is typically used during inference or testing, where the model 
            # is not being trained and no gradients need to be computed. 
            # In evaluation mode, certain layers such as dropout and batch normalization 
            # behave differently than during training mode (`model.train()`), 
            # which can affect the model's output. 
            # Therefore, it is important to set the model to the appropriate mode depending on 
            # whether it is being trained or evaluated. 
            self.dqn_model.eval()
          
            # `.max(1)` is a PyTorch function that returns the maximum value of a tensor 
            # along a specific dimension. 
            # In this case, it is used to get the maximum Q-value for each next state in the batch of experiences. 
            # The argument `1` specifies that the maximum should be taken along the second dimension of the tensor.
            #`.unsqueeze(1)` is a PyTorch function that adds a new dimension to a tensor at the specified position. 
            # In this case, it is used to add a batch dimension to the next state tensor, 
            # which is required as input to the DQN model. 
            # The argument `1` specifies that the new dimension should be added at position 1.

            # your implementation for next_q_values with `max(1)` and `unsqueeze` 
            next_q_values = self.dqn_model(next_state_tensor).max(1)[0].unsqueeze(1)

        self.dqn_model.train()

        q_values = self.dqn_model(state_tensor).gather(1, action_tensor)
        
        # your implementation for expected_q_values based on reward_tensor, next_q_values, done_tensor
        expected_q_values = reward_tensor + self.gamma * next_q_values * (1 - done_tensor)

        # your implementation for loss of current q value and target q value based on  'F.smooth_l1_loss'
        loss = F.smooth_l1_loss(q_values, expected_q_values)

        self.optimizer.zero_grad()
        loss.mean().backward()
        self.optimizer.step()
        self.loss = loss.detach().numpy()
    
    def learn(self, batch_size):
        """
        Trains the DQN model using a batch of experiences from memory.
        """
        batch = self.memory.get_batch(batch_size)
        self.learn_from_memory(batch)

    def set_optimizer(self, optimizer):
        """
        Sets the optimizer for the DQN model.
        """
        self.optimizer = optimizer

    def set_memory(self, memory):
        """
        Sets the memory for the DQN agent.
        """
        self.memory = memory

    def set_batch_size(self, batch_size):
        """
        Sets the batch size for the DQN agent.
        """
        self.batch_size = batch_size


In [ ]:
from IPython.display import clear_output
import time
def ExperimentDQNAgent():
    grid_world = GridWorld(10, 10, blocked_positions=[(1, 1), (2, 2), (3, 3), (4, 4), (3, 2), (3, 1), (5,0),
                                                        (5, 5),  (8, 9),  (8, 8),  (8, 7),])
    dqn_model = DQN(grid_world.observation_space, grid_world.action_space)
    agent = DQNAgent(alpha=0.001, gamma=0.95, epsilon=0.99, dqn_model=dqn_model)
    optimizer = optim.Adam(dqn_model.parameters(), lr=0.001)
    memory = ReplayMemory(capacity=50000)
    agent.set_optimizer(optimizer)
    agent.set_memory(memory)
    agent.set_batch_size(128) 

    num_episodes = 200
    batch_size = 128
    step_per_episode = []
    cumulated_reward_per_episode = []
    loss = []
    
    #warm up, save examples into memory
    print("Warm up..")
    for episode in range(1000):    
        grid_world.reset()
        step=0
        while not grid_world.is_game_over() and step < 500:
            step += 1
            state = grid_world.get_state()
            agent.dqn_model.eval()
            action = agent.get_action(state)
            grid_world.make_action(action)
            next_state = grid_world.get_state()
            reward = grid_world.get_reward()
            done = grid_world.is_game_over()
            agent.memory.push(state, action, reward, next_state, done)
            state = next_state
        if reward > 0:
            agent.memory.add_episode_pos()    

    print(len(agent.memory.memory), len(agent.memory.pos_memory), len(agent.memory.neg_memory), )

    #learning
    print("Training..")
    for episode in range(num_episodes):
        if episode % 10 == 0:
            print("episode_", episode)
        if episode % 10 == 0:
            torch.save(agent.dqn_model.state_dict(), f'grid_world_learned_policy_dqn_{episode}.pt')
        grid_world.reset()
        cumulated_reward = 0
        step=0
        q_values_of_state=[]
        while not grid_world.is_game_over() and step < 500:
            step += 1
            state = grid_world.get_state()
            agent.dqn_model.eval()
            action = agent.get_action(state)
            grid_world.make_action(action)
            next_state = grid_world.get_state()
            reward = grid_world.get_reward()
            done = grid_world.is_game_over()
            #print(state, action, reward)
            q_values_of_state.append(agent.get_q_value(state).numpy())
            agent.memory.push(state, action, reward, next_state, done)
            state = next_state
            cumulated_reward += reward    
            agent.dqn_model.train()
            agent.learn(batch_size=batch_size)
        cumulated_reward_per_episode.append(cumulated_reward)
        step_per_episode.append(step)
        print(f"episode {episode}, cumulated_reward {cumulated_reward:.02f}, step {step}, loss {agent.loss:.05f}, epsilon {agent.epsilon:.02f}")
        
        if reward > 0:
            agent.memory.add_episode_pos()    
            
        agent.update_epsilon(0.98)
         
    
    plt.plot(cumulated_reward_per_episode)
    plt.title('Cumulated reward per episode')
    plt.xlabel('Episode')
    plt.ylabel('Cumulated reward')
    plt.show()

    plt.plot(step_per_episode)
    plt.title('Step per episode')
    plt.xlabel('Episode')
    plt.ylabel('Step')
    plt.show()

ExperimentDQNAgent()


In [ ]:
def SimulationDQNAgent(episode=180):
    grid_world = GridWorld(10, 10, blocked_positions=[(1, 1), (2, 2), (3, 3), (4, 4), (3, 2), (3, 1), (5,0),
                                                        (5, 5),  (8, 9),  (8, 8),  (8, 7),])
    
    # Load the learned policy
    dqn_model = DQN(grid_world.observation_space, grid_world.action_space)
    
    dqn_model.load_state_dict(torch.load(f'grid_world_learned_policy_dqn_{episode}.pt'))
    agent = DQNAgent(alpha=0.5, gamma=0.9, epsilon=0.3, dqn_model=dqn_model)
    agent.dqn_model.eval()

    # Set up the display
    screen = pygame.display.set_mode([(CELL_SIZE + MARGIN) * grid_world.width + MARGIN, (CELL_SIZE + MARGIN) * grid_world.height + MARGIN])
    pygame.display.set_caption("GridWorld")

    # Main loop
    running = True
    clock = pygame.time.Clock()
    visited_positions = []
    while not grid_world.is_game_over():
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        # Use learned policy for agent simulation
        if not grid_world.is_game_over():
            # Update the game state
            state = grid_world.get_state()
            action = agent.get_action(state)
            grid_world.make_action(action)

            visited_positions.append(grid_world.player_position)
        
        # Draw the grid
        screen.fill(BLACK)
        draw_grid(screen, grid_world)

        # Draw visited positions with a changing color from dense to transparent
        for i, pos in enumerate(visited_positions):
            x, y = pos
            alpha = int(100 + 155 * (i / len(visited_positions)))
            color = (255, 200, 200, alpha)  # Light red with changing transparency
            temp_surface = pygame.Surface((CELL_SIZE, CELL_SIZE), pygame.SRCALPHA)
            pygame.draw.rect(temp_surface, color, [0, 0, CELL_SIZE, CELL_SIZE])
            screen.blit(temp_surface, [(MARGIN + CELL_SIZE) * x + MARGIN, (MARGIN + CELL_SIZE) * y + MARGIN])

        # Update the display
        pygame.display.flip()

        # Limit the frame rate
        clock.tick(10)

    # print the q value table and action table  
    q_table = np.zeros([grid_world.width, grid_world.height])
    action_table = np.zeros([grid_world.width, grid_world.height], dtype=str)
    for i in range(grid_world.width):
        for j in range(grid_world.height):
            pos = (i,j)
            state = grid_world.get_state()
            state[0] = 0
            state[0][pos] = 1
            q_table[j][i] =agent.get_q_value(state).numpy().max().round(2)
            action_table[j][i] = grid_world.actions[np.argmax(agent.get_q_value(state).numpy())]

    with np.printoptions(precision=3, suppress=True):
        print("q_table\n", q_table)
        print("action_table\n", action_table)
    
    # Quit pygame
    pygame.quit()

SimulationDQNAgent(episode=190)